# S&P 500 feature backfill directly to S3
Calculates daily features ONCE, extracts eligible Wednesday snapshots, and uploads one Snappy-compressed Parquet file per empty date folder. No Kafka/EC2 broker required.

Destination: s3://s3-stock-market-project-ashwin/features/feature_version=close_features_v1/data_version=kaggle_2026_02/week_date=YYYY-MM-DD/features.parquet

Existing date folders containing any non-folder object are skipped, including 2025-06-18 if its JSON files exist. Skipping does NOT certify date completeness; inspect the skip report (especially partial JSON counts). No existing objects are deleted or replaced. Run one backfill writer; stop other producers/consumers writing this prefix during backfill.

For Athena: your old JSON date is preserved and new dates are Parquet. Do not point a Parquet-only table indiscriminately at the mixed root. Query format-specific partitions/tables, or later convert the old JSON into a separate uniform Parquet dataset. This notebook does not alter Athena.

First authenticate in your Mac terminal: `aws sso login --profile admin`.



In [ ]:
%pip install pandas numpy pandas_market_calendars pyarrow 'boto3>=1.36'


In [ ]:
from pathlib import Path
CSV_PATH = Path("SP500_Historical_Data.csv")
BUCKET = "s3-stock-market-project-ashwin"
PREFIX = (
    "features_parquet/"
    "feature_version=close_features_v1/"
    "data_version=kaggle_2026_02/"
)
FEATURE_VERSION = "close_features_v1"
DATA_VERSION = "kaggle_2026_02"
AWS_PROFILE = "admin"
AWS_REGION = "ap-southeast-2"
AS_OF = None  # None = final CSV date; or "2026-02-18" (complete end-of-day data)
START_WEEK = None  # optional lower bound, e.g. "2020-01-01"; history still used for features
DRY_RUN = True  # review plan first, then False and rerun the LAST cell



## Feature computation
Same 12 close-only features and eligibility as the one-date notebook. Other OHLCV columns are unused.
Approximate one-year warm-up is required; early weeks will not have eligible rows.
Holiday Wednesdays use the preceding NYSE session. Missing individual quotes are not carried forward.
Research training labels remain local and are never uploaded with feature snapshots.
Historical feature values use only prior/current prices, but the source universe still has survivorship bias.


In [ ]:
#!/usr/bin/env python3
"""Close-based weekly features for SP500_Historical_Data.csv.

Install: pip install pandas numpy pandas_market_calendars
Run: python build_sp500_features.py --input SP500_Historical_Data.csv
Optional: --as-of 2026-02-18 --output features_20260218

Assumes complete end-of-day data through --as-of (default: max CSV date).
Wednesday anchors use the last NYSE session on/before Wednesday. A missing
stock quote on that session stays missing; it is NOT a holiday fallback.
Training is X(t) -> adjusted-close return from t to t+1; inference is X(latest).
These are research labels, NOT executable returns after observing close t.
For an executable backtest, separately align entry/exit prices after signals.
No survivorship correction or sector neutralization is performed.
Outputs are CSV plus metadata.json. Pass ONLY metadata['feature_columns'] to ML.
"""
import argparse
import json
from pathlib import Path

import numpy as np
import pandas as pd
import pandas_market_calendars as mcal


def compute_features(input_path, as_of=None):
    df = pd.read_csv(input_path, usecols=['Ticker', 'Date', 'Adj Close'])
    if df.empty or df[['Ticker', 'Date']].isna().any().any():
        raise ValueError('Empty data or missing ticker/date.')
    df['Ticker'] = df['Ticker'].astype(str).str.strip()
    df['Date'] = pd.to_datetime(df['Date'], errors='raise').dt.normalize()
    if (df['Ticker'] == '').any() or df.duplicated(['Ticker', 'Date']).any():
        raise ValueError('Blank ticker or duplicate ticker/date: fix input first.')
    df['Adj Close'] = pd.to_numeric(df['Adj Close'], errors='raise')
    invalid = df['Adj Close'].isna() | ~np.isfinite(df['Adj Close']) | (df['Adj Close'] <= 0)
    if invalid.any():
        raise ValueError(f'{invalid.sum()} invalid adjusted prices: investigate first.')
    source_max = df['Date'].max()
    cutoff = pd.Timestamp(as_of).normalize() if as_of else source_max
    if cutoff < df['Date'].min():
        raise ValueError('As-of precedes the data.')
    df = df.loc[df['Date'] <= cutoff].copy()
    # Include a buffer so an early holiday anchor can find its preceding session.
    sessions = mcal.get_calendar('NYSE').valid_days(
        start_date=df['Date'].min() - pd.Timedelta(days=10), end_date=cutoff
    ).tz_localize(None)
    if not df['Date'].isin(sessions).all():
        raise ValueError('Input contains dates outside the NYSE session calendar.')
    anchors = pd.DataFrame({'week_date': pd.date_range(df['Date'].min(), cutoff, freq='W-WED')})
    if anchors.empty:
        raise ValueError('No completed Wednesday anchors in this range.')
    calendar = pd.merge_asof(
        anchors, pd.DataFrame({'price_date': sessions}),
        left_on='week_date', right_on='price_date', direction='backward'
    )
    if calendar['price_date'].iloc[-1] > df['Date'].max():
        raise ValueError('Latest required snapshot is after available data; update CSV or reduce --as-of.')
    calendar['label_end_week'] = calendar['week_date'] + pd.Timedelta(days=7)
    calendar['label_end_date'] = calendar['price_date'].shift(-1)
    feature_columns = [
        'return_1w', 'return_4w', 'return_13w', 'return_26w', 'momentum_12_1',
        'volatility_20d', 'volatility_60d', 'price_to_sma20', 'price_to_sma60',
        'bollinger_z20', 'rsi14_simple', 'close_to_high252',
    ]
    parts = []
    missing_sessions = 0
    for ticker, group in df.groupby('Ticker', sort=True):
        # Reindex to sessions: rolling windows cannot silently skip missing days.
        grid = sessions[(sessions >= group['Date'].min()) & (sessions <= cutoff)]
        p = group.set_index('Date')['Adj Close'].reindex(grid)
        missing_sessions += int(p.isna().sum())
        r = p.pct_change(fill_method=None)
        daily = pd.DataFrame({'price_date': grid, 'adj_close': p.to_numpy()})
        mean20, std20 = p.rolling(20).mean(), p.rolling(20).std()
        daily['volatility_20d'] = (r.rolling(20).std() * np.sqrt(252)).to_numpy()
        daily['volatility_60d'] = (r.rolling(60).std() * np.sqrt(252)).to_numpy()
        daily['price_to_sma20'] = (p / mean20 - 1).to_numpy()
        daily['price_to_sma60'] = (p / p.rolling(60).mean() - 1).to_numpy()
        z = (p - mean20) / std20.replace(0, np.nan)
        daily['bollinger_z20'] = z.mask(std20.eq(0), 0).to_numpy()
        delta = p.diff()
        gain = delta.clip(lower=0).rolling(14).mean()
        loss = (-delta.clip(upper=0)).rolling(14).mean()
        # Simple rolling RSI, deliberately not Wilder-smoothed RSI.
        rsi = 100 * gain / (gain + loss)
        daily['rsi14_simple'] = rsi.mask((gain + loss).eq(0), 50).to_numpy()
        daily['close_to_high252'] = (p / p.rolling(252).max() - 1).to_numpy()
        weekly = calendar.merge(daily, on='price_date', how='left', validate='one_to_one')
        weekly.insert(0, 'Ticker', ticker)
        for weeks in (1, 4, 13, 26):
            weekly[f'return_{weeks}w'] = weekly['adj_close'] / weekly['adj_close'].shift(weeks) - 1
        # Approximate 12-minus-1-month momentum using exact 52/4 weekly anchors.
        weekly['momentum_12_1'] = weekly['adj_close'].shift(4) / weekly['adj_close'].shift(52) - 1
        weekly['target_return_1w'] = weekly['adj_close'].shift(-1) / weekly['adj_close'] - 1
        parts.append(weekly)
    panel = pd.concat(parts, ignore_index=True).sort_values(['week_date', 'Ticker'])
    panel[feature_columns] = panel[feature_columns].replace([np.inf, -np.inf], np.nan)
    eligible = panel[feature_columns].notna().all(axis=1) & panel['adj_close'].notna()
    latest = calendar['week_date'].iloc[-1]
    training = panel.loc[eligible & panel['target_return_1w'].notna() &
                         panel['label_end_week'].le(latest)].copy()
    # Equal outcomes get equal relevance. Categories can be empty in small groups.
    pct = training.groupby('week_date')['target_return_1w'].rank(method='average', pct=True)
    training['target_relevance'] = np.minimum(np.ceil(pct * 10) - 1, 9).astype(int)
    inference_columns = ['Ticker', 'week_date', 'price_date', 'adj_close'] + feature_columns
    inference = panel.loc[eligible & panel['week_date'].eq(latest), inference_columns].copy()
    if inference.empty:
        raise ValueError('No eligible inference rows. Need about 52 weeks of history and complete windows.')
    return panel, training, inference, {"feature_columns": feature_columns, "as_of": str(cutoff.date()), "inference_week": str(latest.date())}



In [ ]:
import time
started = time.perf_counter()
weekly_panel, training, inference, metadata = compute_features(CSV_PATH, AS_OF)
FEATURE_COLUMNS = metadata["feature_columns"]
eligible = weekly_panel[FEATURE_COLUMNS].notna().all(axis=1) & weekly_panel["adj_close"].notna()
columns = ["Ticker", "week_date", "price_date", "adj_close"] + FEATURE_COLUMNS
snapshots = weekly_panel.loc[eligible, columns].copy()
if START_WEEK is not None:
    snapshots = snapshots.loc[snapshots["week_date"].ge(pd.Timestamp(START_WEEK))].copy()
if snapshots.empty:
    raise ValueError("No eligible snapshots in the chosen range.")
snapshots["week_date"] = snapshots["week_date"].dt.strftime("%Y-%m-%d")
snapshots["price_date"] = snapshots["price_date"].dt.strftime("%Y-%m-%d")
snapshots.insert(0, "data_version", DATA_VERSION)
snapshots.insert(0, "feature_version", FEATURE_VERSION)
assert not snapshots.duplicated(["week_date", "Ticker"]).any()
print(f"Computed once in {time.perf_counter()-started:.1f}s")
print(f"{len(snapshots):,} rows, {snapshots['week_date'].nunique():,} dates")
display(snapshots.head())



## S3 inspection and upload helpers
One paginated listing discovers existing objects. A single PUT per new date replaces hundreds of per-stock requests. Conditional writes prevent overwriting an existing features.parquet. Reruns discover successful uploads and skip them. If another writer creates a different object in the same date concurrently, a conditional PUT cannot prevent a mixed folder; run this as the sole writer.


In [ ]:
import io
import re
import boto3
from botocore.exceptions import ClientError
from collections import defaultdict
from urllib.parse import unquote

def inspect_existing(s3, bucket, prefix):
    existing = defaultdict(list)
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if key.endswith("/"):
                continue
            relative = key[len(prefix):]
            match = re.match(r"week_date=(\d{4}-\d{2}-\d{2})/(.+)$", relative)
            if match:
                existing[match.group(1)].append(match.group(2))
    return dict(existing)

def make_plan(snapshots, existing):
    rows = []
    for date, group in snapshots.groupby("week_date", sort=True):
        files = existing.get(date, [])
        json_tickers = {unquote(f[:-5]) for f in files if f.endswith(".json") and "/" not in f}
        expected = set(group["Ticker"])
        rows.append({
            "week_date": date, "expected_stocks": len(group),
            "existing_objects": len(files),
            "action": "skip_existing" if files else "upload",
            "missing_expected_json_tickers": len(expected - json_tickers) if json_tickers else None,
        })
    return pd.DataFrame(rows)

def backfill(s3, bucket, prefix, snapshots, dry_run=True):
    # Refresh on every invocation, including after an interrupted upload run.
    existing = inspect_existing(s3, bucket, prefix)
    plan = make_plan(snapshots, existing)
    if dry_run:
        return plan
    results = []
    for date, group in snapshots.groupby("week_date", sort=True):
        if existing.get(date):
            results.append({"week_date": date, "status": "skipped_existing"})
            continue
        key = f"{prefix}week_date={date}/features.parquet"
        buffer = io.BytesIO()
        group.to_parquet(buffer, index=False, engine="pyarrow", compression="snappy")
        payload = buffer.getvalue()
        try:
            s3.put_object(
                Bucket=bucket, Key=key, Body=payload,
                ContentType="application/vnd.apache.parquet",
                IfNoneMatch="*",
            )
        except ClientError as exc:
            if str(exc.response["Error"]["Code"]) in ("PreconditionFailed", "412"):
                results.append({"week_date": date, "status": "skipped_existing_key"})
                continue
            print(f"Stopped at {date}. Earlier successful dates can be skipped on rerun.")
            raise
        results.append({"week_date": date, "status": "uploaded",
                        "rows": len(group), "bytes": len(payload)})
        if sum(r["status"] == "uploaded" for r in results) % 50 == 0:
            print(f"Progress through {date}: {len(results)} dates processed")
    return pd.DataFrame(results)



## Preview, then upload
With DRY_RUN=True this cell only lists S3 objects and displays the plan.
Check that 2025-06-18 is skipped. For a JSON-only date, a nonzero missing_expected_json_tickers count suggests a partial/inconsistent snapshot; it remains untouched as requested.
Then set DRY_RUN=False above and rerun this cell. Do not rerun feature computation.
Snapshots are feature-only, compatible with constructing labels later by joining exact adjacent weekly anchors. Do not shift across a missing week.


In [ ]:
session = boto3.Session(profile_name=AWS_PROFILE, region_name=AWS_REGION)
s3 = session.client("s3")
started = time.perf_counter()
report = backfill(s3, BUCKET, PREFIX, snapshots, dry_run=DRY_RUN)
display(report.head(10))
display(report.loc[report["week_date"].eq("2025-06-18")])
display(report["action" if DRY_RUN else "status"].value_counts())
if DRY_RUN:
    display(report.loc[report["action"].eq("skip_existing")])
    print("Preview only. Set DRY_RUN=False and rerun this cell to upload.")
print(f"Elapsed: {time.perf_counter()-started:.1f}s")



References: [S3 paginated listing](https://docs.aws.amazon.com/boto3/latest/reference/services/s3/paginator/ListObjectsV2.html), [S3 put_object](https://docs.aws.amazon.com/boto3/latest/reference/services/s3/client/put_object.html).
